## Exercícios

> Retirados de [learn-python: sqlalchemy_orm-questions](https://aviadr1.github.io/learn-advanced-python/11_db_access/exercise/sqlalchemy_orm-questions.html).

#### Q1.

Baixa e extraia o arquivo compactado com o banco de dados [Chinook database](https://www.sqlitetutorial.net/sqlite-sample-database/). Salve o arquivo `chinook.db` na mesma pasta deste script.
* Link para baixar: http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip

<img width=500 src=https://www.sqlitetutorial.net/wp-content/uploads/2015/11/sqlite-sample-database-color.jpg>


#### Q2.

Leia o código e os comentários das células a seguir para entender como acessamos os modelos ORM de um banco já existente.

In [ ]:
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.orm import Session

engine = create_engine("sqlite+pysqlite:///chinook.db", echo=False)

### extrai as classes da base de dados Chinook
metadata = MetaData()
metadata.reflect(engine)

# O metadata tem informações sobre as tabelas
# que serão usadas para criar os modelos ORM
for table_name, table in metadata.tables.items():
    print(table_name)
    print(table.columns.keys())
    print(table.columns.items())
    print('-'*25)

### configura o objeto Base mapeando os modelos ORM das tabelas
from sqlalchemy.ext.automap import automap_base
Base = automap_base(metadata=metadata)
Base.prepare()

# o objeto Base tem os modelos ORM que podemos usar
# para manipular o banco de dados
print(Base.classes.items())

In [ ]:
# A seguir um exemplo de query na tabela Albums
# usamos o objeto Base para acessar cada modelo ORM.

session = Session(engine)
res = session.scalars(select(Base.classes.albums))
first_album = res.first()
print(first_album.AlbumId, first_album.Title)

#### Q3. 
Com base nos códigos anteriores realize as operações solicitadas nas células a seguir:


In [10]:
### Imprima os três primeiros registros da tabela tracks

res = session.scalars(select(Base.classes.tracks).limit(3))

for track in res:
    print(track.TrackId, track.Name, track.AlbumId, track.Composer)

1 For Those About To Rock (We Salute You) 1 Angus Young, Malcolm Young, Brian Johnson
2 Balls to the Wall 2 None
3 Fast As a Shark 3 F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman


In [11]:
### Imprima o nome da faixa e o título do álbum das primeiras 20 faixas na tabela tracks.

tracks = Base.classes.tracks
albums = Base.classes.albums

res = session.execute(
    select(tracks.Name, albums.Title)
    .join(albums, tracks.AlbumId == albums.AlbumId)
    .limit(20)
)

for nome_faixa, titulo_album in res:
    print(nome_faixa, "-", titulo_album)

For Those About To Rock (We Salute You) - For Those About To Rock We Salute You
Put The Finger On You - For Those About To Rock We Salute You
Let's Get It Up - For Those About To Rock We Salute You
Inject The Venom - For Those About To Rock We Salute You
Snowballed - For Those About To Rock We Salute You
Evil Walks - For Those About To Rock We Salute You
C.O.D. - For Those About To Rock We Salute You
Breaking The Rules - For Those About To Rock We Salute You
Night Of The Long Knives - For Those About To Rock We Salute You
Spellbound - For Those About To Rock We Salute You
Balls to the Wall - Balls to the Wall
Fast As a Shark - Restless and Wild
Restless and Wild - Restless and Wild
Princess of the Dawn - Restless and Wild
Go Down - Let There Be Rock
Dog Eat Dog - Let There Be Rock
Let There Be Rock - Let There Be Rock
Bad Boy Boogie - Let There Be Rock
Problem Child - Let There Be Rock
Overdose - Let There Be Rock


In [12]:
### Imprima as 10 primeiras vendas de faixas da tabela invoice_items
### Para essas 10 primeiras vendas, imprima os nomes das faixas vendidas e a quantidade vendida.

invoice_items = Base.classes.invoice_items
tracks = Base.classes.tracks

res = session.execute(
    select(
        invoice_items.InvoiceLineId,
        tracks.Name,
        invoice_items.Quantity
    )
    .join(tracks, invoice_items.TrackId == tracks.TrackId)
    .limit(10)
)

for venda, faixa, quantidade in res:
    print(venda, "-", faixa, "-", quantidade)


1 - Balls to the Wall - 1
2 - Restless and Wild - 1
3 - Put The Finger On You - 1
4 - Inject The Venom - 1
5 - Evil Walks - 1
6 - Breaking The Rules - 1
7 - Dog Eat Dog - 1
8 - Overdose - 1
9 - Love In An Elevator - 1
10 - Janie's Got A Gun - 1


In [13]:
### Imprima os nomes das 10 faixas mais vendidas e quantas vezes foram vendidas.

invoice_items = Base.classes.invoice_items
tracks = Base.classes.tracks

res = session.execute(
    select(
        tracks.Name,
        func.sum(invoice_items.Quantity).label("total_vendas")
    )
    .join(invoice_items, tracks.TrackId == invoice_items.TrackId)
    .group_by(tracks.Name)
    .order_by(func.sum(invoice_items.Quantity).desc())
    .limit(10)
)

for faixa, vendas in res:
    print(faixa, "-", vendas, "vendas")

The Trooper - 5 vendas
Untitled - 4 vendas
The Number Of The Beast - 4 vendas
Sure Know Something - 4 vendas
Hallowed Be Thy Name - 4 vendas
Eruption - 4 vendas
Where Eagles Dare - 3 vendas
Welcome Home (Sanitarium) - 3 vendas
Sweetest Thing - 3 vendas
Surrender - 3 vendas


In [14]:
### Quem são os 10 artistas que mais venderam?
### dica: você precisa juntar as tabelas invoice_items, tracks, albums e artists

invoice_items = Base.classes.invoice_items
tracks = Base.classes.tracks
albums = Base.classes.albums
artists = Base.classes.artists

res = session.execute(
    select(
        artists.Name,
        func.sum(invoice_items.Quantity).label("total_vendas")
    )
    .join(albums, artists.ArtistId == albums.ArtistId)
    .join(tracks, albums.AlbumId == tracks.AlbumId)
    .join(invoice_items, tracks.TrackId == invoice_items.TrackId)
    .group_by(artists.Name)
    .order_by(func.sum(invoice_items.Quantity).desc())
    .limit(10)
)

for artista, vendas in res:
    print(artista, "-", vendas, "vendas")

Iron Maiden - 140 vendas
U2 - 107 vendas
Metallica - 91 vendas
Led Zeppelin - 87 vendas
Os Paralamas Do Sucesso - 45 vendas
Deep Purple - 44 vendas
Faith No More - 42 vendas
Lost - 41 vendas
Eric Clapton - 40 vendas
R.E.M. - 39 vendas
